# 因子分析 Notebook

用于探索和可视化因子表现。

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']

from data.fetcher import DataFetcher
from factor.technical import TechnicalFactors
from strategy.model import FactorModel
from backtest.metrics import compute_metrics

print('环境就绪')

## 1. 数据概览

拉取沪深300成分股日线数据。

In [ ]:
fetcher = DataFetcher()
stocks = fetcher.get_universe('hs300')[:5]
print(f'样本股票: {stocks}')

# 拉取单只股票数据
df = fetcher.get_daily(stocks[0])
df.head()

## 2. 因子分布

计算技术因子并查看分布。

In [ ]:
# 构建多股票 close DataFrame
close_data = {}
for sym in stocks:
    df = fetcher.get_daily(sym)
    if 'close' in df.columns:
        close_data[sym] = df['close']

close_df = pd.DataFrame(close_data).sort_index()
print(f'数据形状: {close_df.shape}')

# 计算因子
tech = TechnicalFactors()
factors = tech.compute({'close': close_df})
print(f'因子数量: {factors.shape[1]}')
factors.describe()

## 3. IC 分析

因子与未来收益的 Rank IC。

In [ ]:
returns = close_df.pct_change()
future_ret = returns.shift(-5)  # 5日后收益

ic_data = []
for col in factors.columns[:10]:  # 前 10 个因子
    f = factors[col]
    # Rank IC
    ic = f.rank(axis=1).corrwith(
        future_ret.rank(axis=1), axis=1
    )
    ic_data.append({'factor': col, 'mean_ic': ic.mean(), 'ic_ir': ic.mean() / ic.std()})

ic_df = pd.DataFrame(ic_data).sort_values('mean_ic', key=abs, ascending=False)
ic_df.head(10)